# 2.5 — Run Third-Party Models in Snowflake

**Exam domain:** Gen AI Functions (Domain 2.0) · **Weight:** 38%

## The problem this solves

Your data science team has a ticket classifier that took three months and beats anything a general model
produces on your data. It currently runs on a laptop, and the scoring job exports a CSV to somebody's
cloud bucket at 6am. Every copy of that data is a governance problem, and every export is a step that can
fail silently.

Running the model inside Snowflake removes the copies. The question is which of the three routes fits:
register the model object, ship a container, or teach a base model your task.

## What you will be able to do

- Log a trained model to the Snowflake Model Registry and call it from SQL and from Python
- Say what `target_platforms` decides, and why it is not simply "CPU or GPU"
- Stand up a Snowpark Container Services service and expose it to SQL through a service function
- Choose between Registry, SPCS and Cortex Fine-tuning, and say what each one costs to own
- Find the limits and the usage views before they find you

## Before you start

- Run `setup/dataset.sql` for `GENAI_STUDY.PUBLIC.SUPPORT_TICKETS`.
- Registry work needs `CREATE MODEL` on the schema (or ownership of it) to log a model, and `USAGE` or
  `READ` on a model to run it.
- SPCS work needs `CREATE COMPUTE POOL` on the account, and `CREATE SERVICE` on the schema plus `USAGE`
  on the compute pool and `READ` on the stage and image repository.

📖 **Snowflake documentation for this notebook**
- [Snowflake Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)
- [Inference from SQL](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)
- [Snowpark Container Services overview](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/overview)
- [CREATE SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-service)
- [CREATE COMPUTE POOL](https://docs.snowflake.com/en/sql-reference/sql/create-compute-pool)
- [Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)


---

## Three routes, and the question that picks between them

| Route | What you hand Snowflake | Good for |
|---|---|---|
| **Snowflake Model Registry** | a trained Python model object | scikit-learn, XGBoost, LightGBM, PyTorch, TensorFlow, Keras, Hugging Face pipelines, Sentence Transformers, MLflow pyfunc, CatBoost, Prophet, and custom code wrapped in a `CustomModel` |
| **Snowpark Container Services** | a Docker image you built | anything that runs in a container — bespoke runtimes, your own inference server, system libraries Snowflake cannot reproduce |
| **Cortex Fine-tuning** | labelled training data | teaching a supported base model your specific task, then calling it like any other Cortex model |

The question that picks between the first two is **packaging, not hardware**. A model logged with
`target_platforms=['SNOWPARK_CONTAINER_SERVICES']` runs on SPCS and can use a GPU, so "Registry means
CPU" is simply not true. What separates them is who owns the image: with the Registry, Snowflake packages
your model object and its declared dependencies; with SPCS, you build, push and maintain the image
yourself.

### Documented Registry limits

- 1,000 versions per model, 10 methods per version, 500 arguments per method
- 15 GB total model size for **warehouse** deployment, and a maximum config file size of 250 KB
- Costs: storage for the artifacts plus warehouse compute for logging and inference. Model serving usage
  is tracked in `SNOWFLAKE.ACCOUNT_USAGE.MODEL_SERVING_USAGE_HISTORY`

→ [More on the Model Registry](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

---
## Part A — Snowflake Model Registry

The Registry is a schema-level home for models. `log_model()` stores the artifact, infers a signature
from sample input, records dependencies, and gives you a versioned object with metrics and lineage.
`target_platforms` then decides where that version can actually run — check the `runnable_in` column of
`SHOW VERSIONS IN MODEL` to see what a given version supports.


In [ ]:
# Step 1: Train a simple sklearn model on our ticket data
from snowflake.snowpark.context import get_active_session
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

session = get_active_session()

# Load training data from Snowflake
df = session.table('GENAI_STUDY.PUBLIC.SUPPORT_TICKETS').to_pandas()
df = df[df['LANGUAGE'] == 'en'].dropna(subset=['TICKET_TEXT', 'CATEGORY'])

X = df['TICKET_TEXT']
y = df['CATEGORY']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Build a TF-IDF + Logistic Regression pipeline
model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=500)),
    ('clf',   LogisticRegression(max_iter=200))
])

model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# Step 2: Log the trained model to the Snowflake Model Registry
from snowflake.ml.registry import Registry
from snowflake.ml.model import custom_model
import pandas as pd

reg = Registry(
    session=session,
    database_name='GENAI_STUDY',
    schema_name='PUBLIC'
)

# Log the model — target_platforms determines where it can be called
mv = reg.log_model(
    model=model,
    model_name='TICKET_CLASSIFIER',
    version_name='v1',
    comment='sklearn TF-IDF + LR ticket classifier trained on SUPPORT_TICKETS',
    # sample_input_data for schema inference
    sample_input_data=pd.DataFrame({'TICKET_TEXT': X_test.head(3).values}),
    # target_platforms decides where this version can run:
    #   'WAREHOUSE' -> runnable in a warehouse, so callable from SQL
    #   'SNOWPARK_CONTAINER_SERVICES' -> served from a container, GPU-capable
    # target_platforms=['WAREHOUSE']   # uncomment to enable warehouse inference
)

print(f"Model logged: {mv.model_name} / {mv.version_name}")
print(f"Available methods: {[m.name for m in mv.show_functions()][:5]}")

In [ ]:
%%sql -r model_inference_1
-- Step 3: call the logged model from SQL.
-- The documented form wraps the model name: MODEL(<name>[, <version_or_alias>])!<method>(...).
-- The version must be runnable in a warehouse -- check `runnable_in` in SHOW VERSIONS IN MODEL.
SELECT
    ticket_id,
    ticket_text,
    MODEL(GENAI_STUDY.PUBLIC.TICKET_CLASSIFIER, LAST)!PREDICT(ticket_text) AS model_prediction,
    category AS actual_category
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
LIMIT 5;


In [ ]:
%%sql -r model_inference_2
-- List models and versions with the SHOW commands:
SHOW MODELS IN SCHEMA GENAI_STUDY.PUBLIC;


In [ ]:
%%sql -r model_inference_3
SHOW VERSIONS IN MODEL GENAI_STUDY.PUBLIC.TICKET_CLASSIFIER;

-- Cost tracking for model serving:
-- SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.MODEL_SERVING_USAGE_HISTORY ORDER BY START_TIME DESC;


In [ ]:
# Model Registry — additional operations (documented Python API)
from snowflake.ml.registry import Registry
from snowflake.snowpark.context import get_active_session

session = get_active_session()
reg = Registry(session=session, database_name='GENAI_STUDY', schema_name='PUBLIC')

# List all models in the schema
print(reg.show_models())

# Fetch a model and one of its versions
m  = reg.get_model('TICKET_CLASSIFIER')
mv = m.version('v1')

# Run inference from Python
# preds = mv.run(test_features, function_name='predict')

# Metrics and metadata
mv.set_metric('test_accuracy', 0.82)
mv.set_metric('f1_score',      0.79)
print(mv.show_metrics())

# Promote a default version
m.default = mv
print('Default version:', m.default.version_name)

# Reminder - target_platforms decides where inference can run:
#   'WAREHOUSE'                    -> runnable in a warehouse, callable from SQL, model <= 15 GB
#   'SNOWPARK_CONTAINER_SERVICES'  -> containerised serving, GPU-capable
#   both                           -> logged for either


> ### ⚠️ Common misconceptions
>
> **"The Model Registry is the CPU option and SPCS is the GPU option."**
> A registered model logged with `target_platforms=['SNOWPARK_CONTAINER_SERVICES']` serves from a
> container and can use a GPU compute pool. The real distinction is who owns the image. Choosing SPCS
> because you need a GPU means taking on image builds, a registry and spec YAML you did not need.
> → [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)
>
> **"`SELECT my_db.my_schema.MY_MODEL!PREDICT(col)` is how you call a model from SQL."**
> The documented form wraps the model name: `SELECT MODEL(<model_name>)!<method>(...)`, or
> `MODEL(<model_name>, <version_or_alias>)!<method>(...)` to pin a version. `LAST` is a useful alias for
> the most recent one.
> → [Inference from SQL](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)
>
> **"There is an `INFORMATION_SCHEMA` view listing my models."**
> Use the `SHOW` commands — `SHOW MODELS IN SCHEMA ...` and `SHOW VERSIONS IN MODEL ...` — or
> `reg.show_models()` from Python. Querying a view that does not exist fails at compile time, which is at
> least a loud failure rather than a quiet one.
> → [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)


---
## Part B — Snowpark Container Services

SPCS runs your own container images inside Snowflake's account boundary. Four objects, in the order you
create them:

```
Image Repository  (an OCI-compliant registry inside Snowflake)
       |
Compute Pool      (a collection of VM nodes -- this is the billing unit)
       |
Service           (long-running containers, defined by a specification YAML)
       |
Service Function  (a SQL-callable entry point into a service endpoint)
```

| Object | SQL | Purpose |
|---|---|---|
| `IMAGE REPOSITORY` | `CREATE IMAGE REPOSITORY ...` | Where you push the Docker image |
| `COMPUTE POOL` | `CREATE COMPUTE POOL ... MIN_NODES = MAX_NODES = INSTANCE_FAMILY =` | Allocates the nodes the service runs on. `SHOW COMPUTE POOL INSTANCE FAMILIES` lists what is available to you; GPU families exist on both AWS and Azure, and some regions have none |
| `SERVICE` | `CREATE SERVICE ... IN COMPUTE POOL ...` | Long-running; Snowflake restarts a container that exits |
| **Job service** | `EXECUTE JOB SERVICE` | The finite-duration counterpart, closer to a stored procedure — it does not restart on exit |
| Service function | `CREATE FUNCTION ... SERVICE = ... ENDPOINT = ... AS '/path'` | Sends batches of rows to a service endpoint from SQL |

Two parameter names are worth memorising because they are easy to half-remember:

- On `CREATE SERVICE`, the file on the stage is named with **`SPECIFICATION_FILE`**, not `SPEC`. The
  inline alternative is `FROM SPECIFICATION $$ ... $$`.
- `AUTO_SUSPEND_SECS` exists on both services and compute pools, with different defaults — 3600 seconds
  on a compute pool, and 0 (disabled) on a service, where a minimum of 300 applies once you enable it.

**The cost you have to actively manage.** A compute pool bills for its nodes while it is running, whether
or not a service on it is handling any traffic. `ALTER COMPUTE POOL ... SUSPEND` stops that, and
`AUTO_SUSPEND_SECS` automates it. This is the single biggest difference from the Registry-plus-warehouse
route, where nothing is billed between queries.

→ [More on SPCS](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/overview) ·
[More on CREATE SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-service)


In [ ]:
%%sql
-- Step 1: create a compute pool. SHOW COMPUTE POOL INSTANCE FAMILIES lists what this
-- account and region can use; GPU families are not available everywhere.
CREATE COMPUTE POOL IF NOT EXISTS GENAI_STUDY_CPU_POOL
    MIN_NODES = 1
    MAX_NODES = 2
    INSTANCE_FAMILY = CPU_X64_S;   -- a GPU family such as GPU_NV_S for GPU workloads
-- Step 2: Create an image repository
CREATE IMAGE REPOSITORY IF NOT EXISTS GENAI_STUDY.PUBLIC.MODEL_IMAGES;


In [ ]:
%%sql -r spcs_setup_2
-- Show the repository URL (use this to push Docker images)
SHOW IMAGE REPOSITORIES IN SCHEMA GENAI_STUDY.PUBLIC;


In [ ]:
%%sql
-- Step 3: Create a stage for the service spec YAML
CREATE STAGE IF NOT EXISTS GENAI_STUDY.PUBLIC.SERVICE_SPECS
    DIRECTORY = (ENABLE = TRUE);


In [ ]:
%%sql -r spcs_setup_4
-- Check compute pool status
SHOW COMPUTE POOLS LIKE 'GENAI_STUDY_CPU_POOL';


In [ ]:
# SPCS service specification YAML - reference structure.
# Upload this to a stage, then name it with SPECIFICATION_FILE in CREATE SERVICE.

spec_yaml = """
spec:
  containers:
    - name: inference-server
      image: <account>.registry.snowflakecomputing.com/genai_study/public/model_images/ticket_classifier:v1
      resources:
        requests:
          memory: 2G
          cpu: 0.5
        limits:
          memory: 4G
          cpu: 1
      env:
        MODEL_PATH: /app/model.pkl
  endpoints:
    - name: predict
      port: 8080
      public: false
  serviceRoles:
    - name: predictor
      endpoints:
        - predict
"""

print("Service Spec YAML structure:")
print(spec_yaml)
print("\nKey fields:")
print("  containers[].image  : fully qualified Docker image path in the image repository")
print("  endpoints[].port    : the port your inference server listens on")
print("  endpoints[].public  : false = internal only; true = externally accessible")
print("  serviceRoles        : controls which Snowflake roles can call each endpoint")

In [ ]:
%%sql
-- Step 4: create the service from a specification file on the stage.
-- The parameter is SPECIFICATION_FILE, not SPEC.
CREATE SERVICE IF NOT EXISTS GENAI_STUDY.PUBLIC.TICKET_INFERENCE_SVC
    IN COMPUTE POOL GENAI_STUDY_CPU_POOL
    FROM @GENAI_STUDY.PUBLIC.SERVICE_SPECS
    SPECIFICATION_FILE = 'ticket_classifier_spec.yaml'
    MIN_INSTANCES = 1
    MAX_INSTANCES = 2;


In [ ]:
%%sql -r spcs_service_2
-- Check service status
SHOW SERVICES IN SCHEMA GENAI_STUDY.PUBLIC;


In [ ]:
%%sql -r spcs_service_3
-- SYSTEM$GET_SERVICE_STATUS is deprecated in favour of SHOW SERVICE CONTAINERS IN SERVICE.
SHOW SERVICE CONTAINERS IN SERVICE GENAI_STUDY.PUBLIC.TICKET_INFERENCE_SVC;


In [ ]:
%%sql
-- Create a SQL-callable service function
CREATE OR REPLACE FUNCTION GENAI_STUDY.PUBLIC.CLASSIFY_TICKET_SPCS(ticket_text VARCHAR)
    RETURNS VARCHAR
    SERVICE = GENAI_STUDY.PUBLIC.TICKET_INFERENCE_SVC
    ENDPOINT = 'predict'
    AS '/predict';


In [ ]:
%%sql -r spcs_service_5
-- Call the service function from SQL
SELECT
    ticket_id,
    GENAI_STUDY.PUBLIC.CLASSIFY_TICKET_SPCS(ticket_text) AS spcs_prediction
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
WHERE language = 'en'
LIMIT 3;


> ### 🤔 Stop and think
>
> - A compute pool bills node-hours until it is suspended; a warehouse bills only while a query runs. For
>   a model scored once an hour in a batch job, which shape fits — and what would change your answer if
>   the model had to answer an interactive request in under a second?
> - The Registry gives you versions, metrics and lineage for free; a container gives you none of that
>   unless you build it. What would you have to log yourself to be able to answer "which model version
>   produced this prediction?" six months from now?
> - Fine-tuning produces a model that is excellent on last quarter's data. Who owns deciding when it has
>   drifted, what evidence would convince them, and what does re-training cost compared to the accuracy
>   it buys back?


---
## Part C — Cortex Fine-tuning, the third route

Sometimes you do not want to bring a model at all. You want a Snowflake-hosted model that is better at
*your* task than the base model is, without owning any serving infrastructure.

Cortex Fine-tuning customises a supported base model with parameter-efficient fine-tuning, driven by a
single function:

```sql
SNOWFLAKE.CORTEX.FINETUNE('CREATE',   <model_name>, <base_model>, <training_query> [, <validation_query> ])
SNOWFLAKE.CORTEX.FINETUNE('SHOW')
SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', <job_id>)
SNOWFLAKE.CORTEX.FINETUNE('CANCEL',   <job_id>)
```

- The operations are `CREATE`, `SHOW`, `DESCRIBE` and `CANCEL`.
- At the time of writing the available base model is `llama3.1-8b`.
- When the job finishes, you call the result like any other model: `AI_COMPLETE('my_tuned_model', ...)`.
- Privileges: `USAGE` on the training data's database, `CREATE MODEL` on (or ownership of) the schema
  that will hold the model, and the `SNOWFLAKE.CORTEX_USER` database role.
- Cost: training is charged on trained tokens — input tokens multiplied by epochs — and inference is
  charged as usual.

**Where it sits on the ladder.** From 2.2: task-specific function → small general model → large general
model → fine-tuned small model → provisioned throughput. Fine-tuning is worth reaching for when a small
model is nearly good enough on a narrow, well-labelled task, because a tuned 8B model can be cheaper at
inference time than a large general model doing the same job. What it costs you is a labelled dataset, a
training run each time the task drifts, and a model that is now yours to evaluate.

→ [More on Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)


---

## Registry, SPCS or fine-tuning — how to decide

| Criterion | Model Registry | SPCS | Cortex Fine-tuning |
|---|---|---|---|
| **What you hand over** | a Python model object | a Docker image you built | labelled training data |
| **Model types** | the supported built-in types, plus custom code in a `CustomModel` | anything containerisable | a supported base model |
| **SQL callable** | yes, `MODEL(<name>)!<method>(...)` when the version is runnable in a warehouse | yes, through a service function | yes, through `AI_COMPLETE` |
| **GPU** | yes, via `target_platforms=['SNOWPARK_CONTAINER_SERVICES']` | yes, via a GPU instance family | managed for you |
| **Dependencies** | `conda_dependencies`, `pip_requirements`, `artifact_repository_map` | full control of the image | none of your concern |
| **Setup effort** | low | high — build, push, spec YAML, compute pool | low, once you have labelled data |
| **Idle cost** | none between queries; warehouse only when invoked | compute pool bills node-hours until suspended | none between calls |
| **Limits to know** | 1,000 versions per model, 10 methods per version, 15 GB for warehouse deployment | container and compute-pool limits | supported base models, and token-based training cost |

---
## Putting it together

**Scenario.** A data science team wants to run a custom fine-tuned LLaMA-2 model for ticket triage. It
needs 16 GB of GPU memory and a custom Python 3.11 environment. Which route?

### Worked solution

**Start by rejecting the wrong question.** "It needs a GPU" does not decide this — both the Registry and
SPCS can reach one. Ask instead whether the model can be expressed as a Python object with dependencies
Snowflake can resolve.

- If it can, log it to the **Model Registry** with `target_platforms=['SNOWPARK_CONTAINER_SERVICES']` and
  serve it from a GPU compute pool. You get versioning, metrics and lineage without building any of it.
- If it cannot — a bespoke inference server, system libraries, a runtime Snowflake's channels do not
  cover — use **SPCS** directly and accept that you now own the image.

Given the stated "custom Python 3.11 environment" that may not be reproducible from Snowflake's conda
channel, **SPCS** is the safe answer here. It is also the more expensive answer, and the reason to state
the assumption out loud: if the dependencies turn out to be ordinary, the Registry route is materially
less to maintain.

**Steps:** build the image → push to the image repository → `CREATE COMPUTE POOL` with a GPU instance
family → upload the specification YAML to a stage → `CREATE SERVICE ... SPECIFICATION_FILE = ...` →
`CREATE FUNCTION ... SERVICE = ... ENDPOINT = ...`.

**Cost control:** compute pools bill node-hours whether or not the service is handling traffic. Suspend
the pool when idle, or set `AUTO_SUSPEND_SECS`.

**The third option nobody asked about.** If "custom fine-tuned LLaMA-2" really means "a model that is
good at our triage task", Cortex Fine-tuning over a supported base model reaches the same outcome with
none of the serving infrastructure. Worth raising before three months of container work.


---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** What are the documented Model Registry limits for versions per model, methods per version, and
model size for warehouse deployment?

<details><summary>Show answer</summary>

A maximum of 1,000 versions per model, 10 methods per version, 500 arguments per method, and a maximum
total model size of 15 GB for warehouse deployment (plus a 250 KB config file limit). The 15 GB figure is
the one that bites: it applies to warehouse deployment, so a model above it has to be served from a
container instead.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**2.** What values does `target_platforms` accept, and what does each one enable?

<details><summary>Show answer</summary>

`WAREHOUSE` and `SNOWPARK_CONTAINER_SERVICES`. The first makes the version runnable in a warehouse, which
is what lets you call it from SQL in an ordinary query; the second serves it from a container, which is
the route to GPU hardware. A version can be logged for both. `SHOW VERSIONS IN MODEL` reports what a
version actually supports in its `runnable_in` column.

→ [Inference from SQL](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)

</details>

**3.** On `CREATE SERVICE`, what is the parameter that names a specification file on a stage?

<details><summary>Show answer</summary>

`SPECIFICATION_FILE = 'my_spec.yaml'`, used with `FROM @stage`. The alternative is `FROM SPECIFICATION
$$ ... $$` with the YAML inline. `SPEC` is a plausible-looking guess and is not the parameter name.

→ [CREATE SERVICE](https://docs.snowflake.com/en/sql-reference/sql/create-service)

</details>

**4.** This fails. Why?

```sql
SELECT GENAI_STUDY.PUBLIC.TICKET_CLASSIFIER!PREDICT(ticket_text) FROM SUPPORT_TICKETS;
```

<details><summary>Show answer</summary>

The documented SQL form wraps the model name in `MODEL(...)`:
`SELECT MODEL(TICKET_CLASSIFIER)!PREDICT(ticket_text) FROM ...`, with an optional second argument to pin
a version or alias, such as `MODEL(TICKET_CLASSIFIER, LAST)`. A second thing to check when a model call
fails is whether the version is runnable in a warehouse at all — a version logged only for container
serving cannot be called this way.

→ [Inference from SQL](https://docs.snowflake.com/en/developer-guide/snowflake-ml/inference/native-batch-inference-sql)

</details>

**5.** A team's SPCS bill is high at the weekend, when nothing queries the service. What is happening?

<details><summary>Show answer</summary>

The compute pool is still running. Pools bill for node-hours regardless of traffic, so an idle service on
a live pool costs the same as a busy one. Fix it with `ALTER COMPUTE POOL ... SUSPEND` or by setting
`AUTO_SUSPEND_SECS`, and note the trade-off: a suspended pool has to resume before it serves the next
request, so the first call after an idle period is slow.

→ [CREATE COMPUTE POOL](https://docs.snowflake.com/en/sql-reference/sql/create-compute-pool)

</details>

**6.** A model was logged without `target_platforms`. The team wants to call it from a SQL query and the
call does not work. Where do you look first?

<details><summary>Show answer</summary>

At `SHOW VERSIONS IN MODEL <name>` and its `runnable_in` column — if `WAREHOUSE` is not listed, the
version cannot be called from a warehouse query and needs to be logged with `WAREHOUSE` in
`target_platforms`. Checking that first avoids a long detour into privileges, which is the other common
cause: using a model requires ownership or `USAGE`/`READ` on it.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**7.** A model is 22 GB. Which serving route is available, and what does that imply?

<details><summary>Show answer</summary>

Not warehouse deployment — the documented maximum total model size for that is 15 GB. It has to be served
from a container, either as a registered model with `target_platforms=['SNOWPARK_CONTAINER_SERVICES']` or
as your own SPCS image. Either way you have moved onto a compute pool, which means node-hour billing and
an idle-cost decision you did not have before.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**8.** What does `CREATE FUNCTION ... SERVICE = ... ENDPOINT = ... AS '/predict'` create, and what is the
`AS` path?

<details><summary>Show answer</summary>

A service function: a SQL-callable UDF bound to one endpoint of a running service, which sends batches of
rows to it over HTTP. The `AS` string is the HTTP path inside your container that handles the request —
so it has to match a route your inference server actually serves. A mismatch here fails at call time, not
at creation time.

→ [Working with services](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)

</details>

**9.** A team needs GPU inference and is about to build a Docker image because "the Registry is CPU only".
What would you tell them?

<details><summary>Show answer</summary>

That the premise is wrong. Logging the model with `target_platforms=['SNOWPARK_CONTAINER_SERVICES']`
serves it from a container on a GPU compute pool, with versioning, metrics and lineage they would
otherwise build themselves. Building the image is the right call only when the model cannot be expressed
as a Python object with resolvable dependencies — a bespoke runtime, unusual system libraries, a custom
inference server.

→ [Model Registry overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/model-registry/overview)

</details>

**10.** A general model classifies your tickets at 78% accuracy and a larger one reaches 88% at four times
the inference cost. What third option is worth pricing, and what does it require?

<details><summary>Show answer</summary>

Cortex Fine-tuning a supported base model such as `llama3.1-8b` on your labelled tickets: a small tuned
model can approach large-model accuracy on a narrow task at small-model inference cost. What it requires
is a labelled dataset good enough to train on, `CREATE MODEL` on the target schema plus `SNOWFLAKE.CORTEX_USER`,
a training cost charged on input tokens times epochs, and an owner for re-training when the ticket mix
changes. If none of that exists, the large model is the honest answer.

→ [Cortex Fine-tuning](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning)

</details>

**11.** Registry or SPCS for a scikit-learn classifier scored nightly over two million rows? Say what each
choice costs.

<details><summary>Show answer</summary>

The Registry. A scikit-learn pipeline is a supported model type, the dependencies are declarable, and
warehouse inference bills only while the nightly job runs — no idle cost at all. SPCS would work and would
add an image to build and patch, a compute pool to suspend, and a spec file to maintain, in exchange for
control nobody in this scenario needs. Choose SPCS here only if the model exceeds 15 GB or needs something
the Python packaging cannot express.

→ [Snowpark Container Services overview](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/overview)

</details>

**12.** *Connecting to another domain.* Your SPCS-served classifier is meant to run inside the enrichment
pipeline from 2.4. What changes about that pipeline, and what new failure modes appear?

<details><summary>Show answer</summary>

The AI function call becomes a service function call, so the task now depends on a compute pool being
resumed and a service being `READY` when it fires. Two failure modes arrive with it: a suspended or
scaling pool makes the first call slow or fails it outright, and a container that crashes leaves the task
erroring on a schedule rather than degrading gracefully. Both argue for `return_error_details`-style
tolerance in the pipeline and for monitoring the service as well as the task — a registered model called
from SQL has neither problem, which is a point in the Registry's favour for pipeline work.

→ [Working with services](https://docs.snowflake.com/en/developer-guide/snowpark-container-services/working-with-services)

</details>
